# Optuna and Hyperopt with Spark ML

# Demo: Optuna and Hyperopt with Spark ML

In this demo, you will learn how to use **Optuna** and **Hyperopt**—powerful hyperparameter optimization (HPO) frameworks—to tune machine learning models in Databricks utilizing Spark ML.

We will demonstrate how to implement these frameworks using a Random Forest Regressor from SparkML, covering:

* **Defining search spaces** for HPO.
* **Creating objective functions** tailored to different frameworks.
* **Optimizing hyperparameters** using two execution strategies:
  * Single-node multithreading for local tuning.
  * Distributed Spark execution for large-scale tuning.

Additionally, we will track and log the results using **MLflow**, enabling efficient management and monitoring of the tuning process.

---

### Distributed Machine Learning in Databricks

Distributing the workload for Hyperparameter tuning with Spark can be broken down into two key components:

1. **Model Training Level:**
   * Utilize PySpark DataFrames for distributed data processing.
   * Leverage Spark ML algorithms, which are inherently scalable.

2. **Optimization Level:**
   * Use distributed computing frameworks (i.e., HyperOpt with Spark Trials) to parallelize multiple training runs.
   * Scale hyperparameter searches efficiently using Spark's distributed environment.

---

> ⚠️ **A Warning Concerning HyperOpt on Databricks**
>
> The open-source version of Hyperopt is no longer being maintained and will be removed in the DBR ML versions 16.0+. This notebook is currently running on a version that supports Hyperopt. Databricks recommends using **Optuna** for single-node optimization or **RayTune** for a similar experience.

# Learning Objectives

By the end of this demo, you will be able to:

### Compare Different Hyperparameter Tuning Frameworks
* **Optuna** for tuning models on a single machine with parallel execution.
* **SparkML + HyperOpt** for fully distributed tuning across a Spark cluster.

### Perform Hyperparameter Tuning using Optuna
* Define an objective function tailored to your model.
* Configure a search space for hyperparameter optimization.
* Optimize hyperparameters using single-node execution.

### Perform Hyperparameter Tuning using HyperOpt with Spark ML
* Implement `CrossValidator` within Spark ML for scalable hyperparameter tuning.
* Utilize **HyperOpt** with **Spark Trials** for parallel optimization.

In [0]:
%pip install -U optuna optuna-integration mlflow
%pip install --upgrade ray[tune]

dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Before starting the demo, run the provided classroom setup script.

In [0]:
#%run ../Includes/Classroom-Setup-02.1

### Other Conventions:

Throughout this demo, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [0]:
#print(f"Username:         {DA.username}")
#print(f"Catalog Name:     {DA.catalog_name}")
#print(f"Schema Name:      {DA.schema_name}")
#print(f"Working Directory: {DA.paths.working_dir}")
#print(f"Dataset Location: {DA.paths.datasets.wine_quality}")

In [0]:
catalog_name= 'workspace'
schema_name='default'
print(f"Catalog Name:     {catalog_name}")
print(f"Schema Name:      {schema_name}")


Catalog Name:     workspace
Schema Name:      default


In [0]:
#workspace.default.`wine-quality-model` # => model
# workspace.default.delta_table_wine06092026 # => tables


## Load Data and Perform Train-Test Split

In this step, we will load the dataset from the Delta table `wine_quality_features`, which is stored in Unity Catalog under:
`{DA.catalog_name}.{DA.schema_name}.wine_quality_features`

### Instructions:

1. **Load the dataset** from the Delta table using `spark.read.table()`.
2. **Split the dataset** into training (80%) and testing (20%) sets to evaluate the model's performance.
   * Since we are using PySpark DataFrames, we will use `.randomSplit()` for the split.

In [0]:
df_teste = spark.read.table(f"{catalog_name}.{schema_name}.delta_table_wine06092026")
df_teste.show(5)

+-------------+----------------+-----------+--------------+---------+-------------------+--------------------+-------+----+---------+-------+-------+
|fixed_acidity|volatile_acidity|citric_acid|residual_sugar|chlorides|free_sulfur_dioxide|total_sulfur_dioxide|density|  pH|sulphates|alcohol|quality|
+-------------+----------------+-----------+--------------+---------+-------------------+--------------------+-------+----+---------+-------+-------+
|          9.3|            0.37|       0.44|           1.6|    0.038|               21.0|                42.0|0.99526|3.24|     0.81|   10.8|      7|
|          9.4|             0.5|       0.34|           3.6|    0.082|                5.0|                14.0| 0.9987|3.29|     0.52|   10.7|      6|
|          9.4|             0.5|       0.34|           3.6|    0.082|                5.0|                14.0| 0.9987|3.29|     0.52|   10.7|      6|
|          7.2|            0.61|       0.08|           4.0|    0.082|               26.0|           

In [0]:
#df = spark.read.table(f"{catalog_name}.{schema_name}.wine_quality_features")
df_2 = spark.read.table(f"{catalog_name}.{schema_name}.wine-quality-model")



In [0]:
import mlflow

# wine-quality-model é um modelo registrado no Unity Catalog (não uma tabela)
# spark.read.table() não funciona com modelos — use MLflow para visualizar
client = mlflow.MlflowClient()
model_info = client.get_registered_model("workspace.default.wine-quality-model")

print(f"Nome: {model_info.name}")
print(f"Descrição: {model_info.description}")
print(f"\nVersões do modelo:")
versions = client.search_model_versions("name='workspace.default.wine-quality-model'")
for v in versions:
    print(f"  Versão {v.version}: status={v.status}, run_id={v.run_id}")

Nome: workspace.default.wine-quality-model
Descrição: 

Versões do modelo:
  Versão 1: status=READY, run_id=6005cc5c9f004680bdcdd5527de15e01


In [0]:
#df = spark.read.table(f"{catalog_name}.{schema_name}.wine_quality_features")
df = spark.read.table(f"{catalog_name}.{schema_name}.delta_table_wine06092026")

# Split the dataset into training and test sets
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

# Part 1: HPO with Optuna and Distributed Training of a Spark Model

In this part, we will use **Optuna** for hyperparameter optimization while training a Spark ML model in parallel.

### How This Works:

* **Optuna runs on a single machine** to manage hyperparameter tuning, where it suggests configurations for each trial and records their performance.
* **Model training can be distributed**, ensuring the ability to scale out and speed up for large datasets and complex models.
* **Each Optuna trial** runs a new model training job on the Spark cluster, allowing it to evaluate different hyperparameter configurations efficiently.

This approach allows us to leverage distributed computing for training while keeping hyperparameter optimization lightweight and efficient on a single node across multiple threads.

---

## Define the Objective Function for Optuna

The first step will be to define the objective function for Optuna. This is the function that Optuna will minimize by optimizing hyperparameters like the number of trees (`numTrees`) and the depth of the tree (`maxDepth`). In our case, the objective function is the **Root Mean Squared Error (RMSE)** since our model is a random forest regressor. However, we will use a distributed training approach within this function by running the training on Spark workers.

### Instructions:

* **Initialize TPESampler Configuration.** In this example we will use Bayesian optimization along with a Gaussian prior to help stabilize the Parzen estimator (known as the Tree-structured Parzen Estimator algorithm).
* **Initialize hyperparameters** using Optuna's `trial.suggest_int()` function. This function samples integers between `low` and `high` for the hyperparameter `<hyperparameter_name>` when calling `trial.suggest_int('<hyperparameter_name>', low, high)`.
* **Train the model** using Spark's distributed cluster by running the `RandomForestRegressor` model on Spark workers.
* **Evaluate the model** using the RMSE metric (`rmse`), and return it as the value to minimize during the optimization. Note, we will tell Optuna to minimize the returned RMSE value when we create an Optuna study later. This happens outside the definition of the objective function.

Refer to the documentation for:
* `optuna.samplers` for the choice of samplers
* `optuna.trial.Trial` for a full list of functions supported to define a hyperparameter search space.

In [0]:
import optuna

optuna_sampler = optuna.samplers.TPESampler(
    consider_prior=True, # Enhance the stability of Parzen estimator by imposing a Gaussian prior when True
    n_startup_trials=3, # The random sampling is used instead of the TPE algorithm until the given number of trials finish in the same study.
    seed=123 # Seed for random number generator.
)

/home/spark-8d6487c7-c5b3-4667-91e7-ab/.ipykernel/74/command-5374862583788002-3162735485:3: FutureWarning: `consider_prior` has been deprecated in v4.3.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.3.0.
  optuna_sampler = optuna.samplers.TPESampler(


In [0]:
from pyspark.ml.regression import RandomForestRegressor, LinearRegression, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

class ObjectiveOptuna:
    """
    Objective function class for Optuna hyperparameter tuning with SparkML models.
    
    Instead of loading the dataset in each trial execution, this class receives
    the training and test datasets during initialization, improving efficiency.
    """
    
    def __init__(self, train_df, test_df, label_column="label"):
        """
        Initializes the objective function with training and test datasets.
        
        Args:
            train_df (DataFrame): Spark DataFrame containing features and label for training.
            test_df (DataFrame): Spark DataFrame containing features and label for evaluation.
            label_column (str): Name of the label column in the dataset. Default is "label".
        """
        self.train_df = train_df
        self.test_df = test_df
        self.label_column = label_column

    def objective_sparkmodel_distributed_Optuna(self, trial):
            """
            Optuna objective function for tuning regression models using SparkML. Possible models are: Linear Regression, Random Forest, and
            Gradient-Boosted Trees.
            
            Args:
                trial (optuna.trial.Trial): An Optuna trial object to suggest hyperparameters.
                
            Returns:
                float: Root Mean Squared Error (RMSE) to minimize.
            """
            
            # Select model type
            model_name = trial.suggest_categorical("model", ["LinearRegression", "RandomForest", "GBTRegressor"])
            
            if model_name == "LinearRegression":
                # Hyperparameter tuning for Linear Regression
                model = LinearRegression(
                    featuresCol="features",
                    labelCol=self.label_column,
                    regParam=trial.suggest_float("reg_param", 0.0, 1.0),
                    elasticNetParam=trial.suggest_float("elastic_net_param", 0.0, 1.0)
                )
                
            elif model_name == "RandomForest":
                # Hyperparameter tuning for Random Forest
                model = RandomForestRegressor(
                    featuresCol="features",
                    labelCol=self.label_column,
                    numTrees=trial.suggest_int("num_trees", 2, 5, log=True),
                    maxDepth=trial.suggest_int("max_depth", 3, 10),
                    minInstancesPerNode=trial.suggest_int("min_instances_per_node", 1, 10)
                )
                
            elif model_name == "GBTRegressor":
                        # Hyperparameter tuning for Gradient-Boosted Trees
                        model = GBTRegressor(
                            featuresCol="features",
                            labelCol=self.label_column,
                            maxDepth=trial.suggest_int("max_depth", 3, 10),
                            maxIter=trial.suggest_int("n_estimators", 2, 5, log=True),
                            stepSize=trial.suggest_float("learning_rate", 0.01, 0.5)
                        )

            # Train the model
            trained_model = model.fit(self.train_df)

            # Generate predictions
            predictions = trained_model.transform(self.test_df)

            # Evaluate performance using RMSE
            rmse = RegressionEvaluator(
                        labelCol=self.label_column,
                        predictionCol="prediction",
                        metricName="rmse"
                    ).evaluate(predictions)

            return rmse

## Optimize The Spark ML model on Single-Machine Optuna and Log Results with MLflow

In this step, we will utilize `MLflow` to track the optimization process by adding out-of-the-box logging provided by Optuna trials using `MLflowCallback()`. Once we have our logging parameters configured, there are two additional steps to take care of before moving onto the run.

1. **Initialize Optuna's `optuna.create_study()`**. A study is corresponds to the optimization task, which is a set of trials and a trial is a process of evaluating an objective function.
2. **Tell Optuna how we want to optimize with `optimize()`**.

Each trial will be logged to MLflow, including the hyperparameters tested and the corresponding `RMSE` values. Optuna will handle the optimization, while training continues to be distributed across Spark workers.

### Instructions:

* **Set up MLflow to track the experiments** using `MLflowCallback()`.
* **Define the storage location** with the variable `storage_url`. In this demonstration, we will be using the driver node to persist our study information, allowing for distributed optimization.
* **Setup an Optuna study** with `optuna.study()`.
* **Optimize hyperparameters** using Optuna's `study.optimize()` method.
* **Log results to MLflow**, including the best hyperparameters and RMSE.
* **End the MLflow run** to ensure that all information is saved.

> **Note on parallelization:** The value of `n_jobs` within the `optimize()` function is the number of parallel jobs. If this argument is set to `-1` (as we have done below), then the number of parallel jobs is set to the number of CPU cores (the default value for this demonstration is 4 cores).

In [0]:
import os
import mlflow
import optuna
from optuna.integration.mlflow import MLflowCallback

# Set up MLflow experiment tracking
experiment_name_spark = os.path.join(
    os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()),
    "02a - Model Tuning with Optuna_Spark"
)

mlflow.set_experiment(experiment_name_spark)
experiment_id_spark = mlflow.get_experiment_by_name(experiment_name_spark).experiment_id

def optuna_hpo_fn(n_trials: int, experiment_id: str, optuna_sampler) -> optuna.study.Study:
    """
    Runs hyperparameter optimization using Optuna with MLflow logging.
    
    Args:
        n_trials (int): Number of trials for optimization.
        experiment_id (str): MLflow experiment ID for logging.
        optuna_sampler (optuna.samplers.BaseSampler): Optuna sampler for search strategy.
        
    Returns:
        optuna.study.Study: The Optuna study object with optimization results.
    """
    
    # MLflow callback to log results
    mlflow_callback_spark = MLflowCallback(
        tracking_uri=mlflow.get_tracking_uri(),
        metric_name="RMSE",
        create_experiment=False,
        mlflow_kwargs={"experiment_id": experiment_id}
    )
    
    # Define the objective function
    objective_function = ObjectiveOptuna(train_df, test_df, label_column="quality").objective_sparkmodel_distributed_Optuna

    # Create or load an Optuna study
    study = optuna.create_study(
        study_name="sparkmodel_optuna_distributed_hpo",
        sampler=optuna_sampler,
        load_if_exists=True,
        direction="minimize"
    )

    # Run optimization
    study.optimize(
        objective_function,
        n_trials=n_trials,
        n_jobs=-1, # Parallel execution
        callbacks=[mlflow_callback_spark]
    )

    # Extract best trial results
    best_trial = study.best_trial
    best_rmse = best_trial.value # RMSE metric

    # Display results
    print(f"Best Trial Number: {best_trial.number}")
    print(f"Best Hyperparameters: {best_trial.params}")
    print(f"Best RMSE: {best_rmse:.4f}")

    # Log the best results manually in MLflow
    with mlflow.start_run(run_name="best_trial_results"):
        mlflow.log_params(best_trial.params)
        mlflow.log_metric("Best RMSE", best_rmse)

    return study # Return study for further analysis


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


In [0]:
# Define the feature columns
feature_columns_rev = [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "pH",
    "sulphates",
    "alcohol",
]


labelCol_rev="quality"



## Execute the Single Node Study

In [0]:
from pyspark.ml.feature import VectorAssembler

# Disable MLflow autologging to prevent unwanted logging of model artifacts
mlflow.autolog(log_models=False, disable=True)

# Assemble feature columns into a single vector column (required by Spark ML models) # adaptado ****
assembler = VectorAssembler(inputCols=feature_columns_rev, outputCol="features")
train_df = assembler.transform(train_df)
test_df = assembler.transform(test_df)

# Invoke Optuna training function on the driver node
# n_jobs=1 (serial) — Spark Connect session is thread-local; RandomForest's
# _call_java("trees") fails in parallel worker threads where session is None
mlflow_callback_spark = MLflowCallback(
    tracking_uri=mlflow.get_tracking_uri(),
    metric_name="RMSE",
    create_experiment=False,
    mlflow_kwargs={"experiment_id": experiment_id_spark}
)

objective_function = ObjectiveOptuna(train_df, test_df, label_column="quality").objective_sparkmodel_distributed_Optuna

study = optuna.create_study(
    study_name="sparkmodel_optuna_distributed_hpo",
    sampler=optuna_sampler,
    load_if_exists=True,
    direction="minimize"
)

study.optimize(
    objective_function,
    n_trials=10,
    n_jobs=1,
    callbacks=[mlflow_callback_spark]
)

best_trial = study.best_trial
best_rmse = best_trial.value

print(f"Best Trial Number: {best_trial.number}")
print(f"Best Hyperparameters: {best_trial.params}")
print(f"Best RMSE: {best_rmse:.4f}")

with mlflow.start_run(run_name="best_trial_results"):
    mlflow.log_params(best_trial.params)
    mlflow.log_metric("Best RMSE", best_rmse)

single_node_study = study

/home/spark-8d6487c7-c5b3-4667-91e7-ab/.ipykernel/74/command-5374862583788056-2097135908:14: FutureWarning: MLflowCallback has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0.
  mlflow_callback_spark = MLflowCallback(
[I 2026-09-15 02:06:40,706] A new study created in memory with name: sparkmodel_optuna_distributed_hpo
[I 2026-09-15 02:06:43,059] Trial 0 finished with value: 0.7767104584402515 and parameters: {'model': 'LinearRegression', 'reg_param': 0.5513147690828912, 'elastic_net_param': 0.7194689697855631}. Best is trial 0 with value: 0.7767104584402515.
[I 2026-09-15 02:06:47,877] Trial 1 finished with value: 0.5988054411903662 and parameters: {'model': 'RandomForest', 'num_trees': 3, 'max_depth': 6, 'min_instances_per_node': 4}. Best is trial 1 with value: 0.5988054411903662.
[I 2026-09-15 02:06:50,313] Trial 2 finished with value: 0.7451254330019597 and parameters: {'model': 'LinearRegression', 'reg_par

Best Trial Number: 9
Best Hyperparameters: {'model': 'RandomForest', 'num_trees': 5, 'max_depth': 8, 'min_instances_per_node': 5}
Best RMSE: 0.5792


## Explanation: Distributing Hyperparameter Tuning and Model Training

The previous cells implemented distributed hyperparameter tuning and training using Optuna, MLflow, and Spark MLlib.

### Key Characteristics of This Setup

| Aspect | Current Implementation |
| :--- | :--- |
| **Hyperparameter Tuning** | Runs on a single machine (Optuna executes locally, even if multiple trials run in parallel). |
| **Parallel Execution** | Trials are parallelized within a single machine and training happens in a distributed fashion across multiple threads. |
| **Database Storage** | Uses default in-memory storage for Optuna trials, limiting multi-machine and multi-process execution. |
| **Experiment Logging** | MLflow logs hyperparameters and RMSE for each trial. |

---

### How to Fully Distribute Hyperparameter Tuning

While this implementation already distributes model training, Optuna's default execution with `n_jobs` utilizes multithreading on a single node, which, due to Python's Global Interpreter Lock, allows for concurrency but not true parallelism in CPU-bound tasks. To achieve true parallelization, Optuna can be configured to use multiprocessing, either on a single node or across multiple nodes, by setting up an appropriate backend such as a relational database. To fully distribute the hyperparameter search across multiple machines:

**Use a centralized database:**

* Within `create_study` include storage such as `storage="sqlite:////local_disk0/optuna_distributed_model.db"` for Multi-processing parallelization with single node or client/server Relational Databases like PostgreSQL or MySQL ex: for Multi-processing parallelization with multiple nodes `storage="mysql://root@localhost/example"`
* This allows multiple workers to share and execute trials.
* **Requirement:** Launch a MySQL instance (can be on AWS RDS, Azure Database for MySQL, GCP Cloud SQL, or an on-prem server). See Optuna Documentation.



# Part 2: Approaches for HPO with SparkML on Databricks

Hyperparameter tuning in a Databricks environment can be challenging due to SparkContext limitations and process forking issues in managed clusters. Below are three recommended approaches to effectively perform hyperparameter tuning while avoiding common pitfalls.

---

### Challenges with Hyperparameter Tuning in Spark and Python

1. **Serialization Issues:**
   * Passing Spark objects (e.g., Spark DataFrame, SparkSession, SparkContext) into a distributed function (Hyperopt or a Spark UDF) can cause failures due to pickling restrictions.

2. **Single SparkContext Per Notebook:**
   * Databricks runs a single Spark driver (the notebook environment) with one Spark session.
   * Workers cannot create new Spark sessions (`SparkSession.getOrCreate()`) without proper master settings.

---

## Recommended Approaches

### Option 1: Use Spark's Built-in Hyperparameter Tuning Tools

**Best for: Native Spark ML hyperparameter tuning**

* **How it Works:**
  * Leverage Spark ML's `CrossValidator` or `TrainValidationSplit` to perform distributed hyperparameter tuning.
  * Spark handles parallelism natively.

* **Pros:**
  * Fully compatible with Databricks.
  * Runs in distributed mode, leveraging Spark Executors.
  * Avoids SparkContext serialization issues.

* **Cons:**
  * Limited to grid search or random search (without custom logic).
  * No advanced Bayesian Optimization (unless implemented manually).


---

### Option 2: Use Hyperopt with SparkTrials for Distributed Tuning

**Best for: Bayesian Optimization on a Spark Cluster**

* **How it Works:**
  * Use **Hyperopt** with `SparkTrials`, which distributes hyperparameter search across Spark Executors.
  * Unlike traditional grid/random search, Bayesian Optimization intelligently selects the best hyperparameters.

* **Pros:**
  * **Bayesian Optimization** (more efficient than exhaustive search).
  * **Parallel execution across Spark Executors** (avoids SparkContext issues).
  * Supports custom ML models beyond Spark ML.

* **Cons:**
  * Requires using Hyperopt's `fmin()` API instead of `CrossValidator`.
  * Does not directly integrate with Spark ML Pipelines (models are trained manually).
  * *Note: The open-source version of Hyperopt is no longer being maintained. Hyperopt will be removed in the next major DBR ML version. Databricks recommends using either Optuna for single-node optimization or RayTune for a similar experience to the deprecated Hyperopt distributed hyperparameter tuning functionality.* [Read more here](https://docs.databricks.com).

## Option 1: Use Spark ML's Built-in Hyperparameter Tuning Tools

In [0]:
import os
import time
import mlflow
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# Required for Spark ML DataFrame caching on serverless/shared compute correção 14/09/2026
os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/default/mlflow_tmp"

label_column = "quality"

# MLflow Experiment Setup
experiment_name_spark_cv = os.path.join(
    os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()),
    "02c - Model Tuning with spark cv"
)

mlflow.set_experiment(experiment_name_spark_cv)
experiment_id_spark_cv = mlflow.get_experiment_by_name(experiment_name_spark_cv).experiment_id

# Ensure feature vectorization
if "features" not in train_df.columns:
    feature_cols = [col for col in train_df.columns if col != label_column]
    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
    train_df_transformed = assembler.transform(train_df).select("features", label_column).na.drop()
else:
    feature_cols = [col for col in train_df.columns if col != label_column]
    train_df_transformed = train_df

# Define RandomForestRegressor and hyperparameter grid
rf = RandomForestRegressor(featuresCol="features", labelCol=label_column, seed=42)


param_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [5, 10, 20])   # Number of trees
    .addGrid(rf.maxDepth, [2, 5, 10])    # Max tree depth
    .build()
)

# Set up CrossValidator
evaluator = RegressionEvaluator(labelCol=label_column, predictionCol="prediction", metricName="rmse")

cv = CrossValidator(
    estimator=rf,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,     # 3-fold cross-validation
    parallelism=4   # Parallel execution
)

with mlflow.start_run(run_name="spark_cv_rf", experiment_id=experiment_id_spark_cv):
    try:
        # Start training
        start_time = time.time()
        cv_model = cv.fit(train_df_transformed)
        training_duration = time.time() - start_time
        mlflow.log_metric("training_duration_s", training_duration)

        # Retrieve best model and evaluate
        best_model = cv_model.bestModel
        train_predictions = best_model.transform(train_df_transformed)
        train_rmse = evaluator.evaluate(train_predictions)
        mlflow.log_metric("train_rmse", train_rmse)
        print(f"Best Model RMSE on training folds: {train_rmse:.4f}")

        # Evaluate on test set
        test_predictions = best_model.transform(test_df)
        test_rmse = evaluator.evaluate(test_predictions)
        mlflow.log_metric("test_rmse", test_rmse)
        print(f"Test RMSE: {test_rmse:.4f}")

        # Log best hyperparameters
        best_num_trees = best_model.getNumTrees
        best_max_depth = best_model.getOrDefault("maxDepth")
        mlflow.log_param("best_numTrees", best_num_trees)
        mlflow.log_param("best_maxDepth", best_max_depth)

        print(f"Best hyperparameters - numTrees={best_num_trees}, maxDepth={best_max_depth}")

        # Log feature importances
        if hasattr(best_model, "featureImportances"):
            importances = best_model.featureImportances
            feat_imp_map = {col: val for col, val in zip(feature_cols, importances.toArray())}
            mlflow.log_text(str(feat_imp_map), "feature_importances.txt")
            print("Feature Importances:", feat_imp_map)

# Log all hyperparameter results
        avg_metrics = cv_model.avgMetrics
        print("\nHyperparameter Combinations and Avg RMSE:")
        print("--------------------------------------------------")
        print(f"{'numTrees':<12}{'maxDepth':<12}{'avg_rmse':<10}")
        print("--------------------------------------------------")

        for i, param_map in enumerate(param_grid):
            avg_rmse = avg_metrics[i] if i < len(avg_metrics) else "N/A"  # Handle index errors safely
            num_trees_val = param_map.get(rf.numTrees, "N/A")
            max_depth_val = param_map.get(rf.maxDepth, "N/A")
            print(f"{num_trees_val:<12}{max_depth_val:<12}{avg_rmse:<10.4f}")

    except Exception as e:
        print(f"Error during cross-validation: {e}")
# End MLflow Run
mlflow.end_run()
print("Cross-validation complete. Check MLflow UI for details.")


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


Best Model RMSE on training folds: 0.3229
Test RMSE: 0.5466
Best hyperparameters - numTrees=20, maxDepth=10
Feature Importances: {'fixed_acidity': np.float64(0.07032844650001865), 'volatile_acidity': np.float64(0.1125291395243702), 'citric_acid': np.float64(0.07785218472815882), 'residual_sugar': np.float64(0.035991069439197595), 'chlorides': np.float64(0.05972089180612271), 'free_sulfur_dioxide': np.float64(0.055920463134892226), 'total_sulfur_dioxide': np.float64(0.08451439290213834), 'density': np.float64(0.0825666955212885), 'pH': np.float64(0.05244083496854777), 'sulphates': np.float64(0.14063290187531874), 'alcohol': np.float64(0.22750297959994645)}

Hyperparameter Combinations and Avg RMSE:
--------------------------------------------------
numTrees    maxDepth    avg_rmse  
--------------------------------------------------
5           2           0.6811    
5           5           0.6444    
5           10          0.6451    
10          2           0.6777    
10          5   

In [0]:
# página 19

## Option 2: Use Hyperopt with SparkTrials for Distributed Tuning

Create a function to define a model to train - here a decision tree regressor - and initialize the first training and RMSE value.

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline
import mlflow
import mlflow.spark

# Set up MLflow experiment tracking
experiment_name_Hyperopt = os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
) + "/02d - Model Tuning with Hyperopt_spark"
mlflow.set_experiment(experiment_name_Hyperopt)
experiment_id_Hyperopt = mlflow.get_experiment_by_name(experiment_name_Hyperopt).experiment_id

def train_tree(minInstancesPerNode, maxBins):
    with mlflow.start_run(nested=True, experiment_id=experiment_id_Hyperopt):
        
        # (A) Assemble features into a single "features" column
        assembler = VectorAssembler(
            inputCols=feature_columns,
            outputCol="features"
        )
        
        # (B) Decision Tree Regressor
        dtr = DecisionTreeRegressor(
            labelCol=label_column,
            featuresCol="features",
            minInstancesPerNode=minInstancesPerNode,
            maxBins=maxBins
        )

        if "features" not in train_df.columns:
            # Combine the stages into a pipeline
            pipeline = Pipeline(stages=[assembler, dtr])
        else:
            train_df_transformed = train_df
            pipeline = Pipeline(stages=[dtr])

        # Train (fit) the pipeline
        model = pipeline.fit(train_df)

        # Evaluate on the test set
        evaluator = RegressionEvaluator(
                    labelCol=label_column,
                    predictionCol="prediction",
                    metricName="rmse"  # You can choose "mse", "mae", or "r2" as well
                )
                
        predictions = model.transform(test_df)
        test_metric = evaluator.evaluate(predictions)

        # Log the RMSE to MLflow
        mlflow.log_metric("test_rmse", test_metric)

    return model, test_metric



mlflow.end_run()

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


In [0]:
feature_columns = feature_columns_rev
initial_model, test_rmse = train_tree(minInstancesPerNode=200, maxBins=2)
print(f"The trained decision tree regressor achieved an RMSE of {test_rmse} on the test data.")

The trained decision tree regressor achieved an RMSE of 0.6619734741026135 on the test data.


In [0]:
#initial_model, test_rmse = train_tree(minInstancesPerNode=200, maxBins=2)
#print(f"The trained decision tree regressor achieved an RMSE of {test_rmse} on the test data.")

Define the train method for use with Hyperopt.

In [0]:
%pip install hyperopt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 973.5/973.5 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 74.2 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

def train_with_hyperopt(params):
    """
    An example train method that calls into Spark MLlib.
    This method is passed to hyperopt.fmin().
    
    :param params: hyperparameters as a dict (consistent with how search space is defined).
    :return: dict with fields 'loss' (scalar loss) and 'status' (success/failure).
    """
    # Convert these to int, since hyperopt may sample floats
    minInstancesPerNode = int(params['minInstancesPerNode'])
    maxBins = int(params['maxBins'])
    
    model, rmse = train_tree(minInstancesPerNode, maxBins)
    
    # For hyperopt, "loss" should be minimized, so we can directly use the RMSE
    loss = rmse
    return {'loss': loss, 'status': STATUS_OK}

Define the range of hyperparameters to explore with the dictionary `space`, choose the optimization algorithm (`algo`), and run the optimization with MLflow logging enabled.

In [0]:
space = {
    'minInstancesPerNode': hp.uniform('minInstancesPerNode', 10, 200),
    'maxBins': hp.uniform('maxBins', 2, 32),
}

algo = tpe.suggest

with mlflow.start_run(run_name="parallel_spark_training_hyperopt", experiment_id=experiment_id_Hyperopt) as parent_run:
    best_params = fmin(
        fn=train_with_hyperopt,
        space=space,
        algo=algo,
        max_evals=8
    )

100%|██████████| 8/8 [00:18<00:00,  2.37s/trial, best loss: 0.6166544409330698]


Get the best parameters and get a final version of the model.

In [0]:
# Print out the parameters that produced the best model
best_params

best_minInstancesPerNode = int(best_params['minInstancesPerNode'])
best_maxBins = int(best_params['maxBins'])

final_model, best_rmse = train_tree(best_minInstancesPerNode, best_maxBins)

Display the RMSE values for the initialized model and the final model after optimization.

In [0]:
# Evaluate both models on the test set for a final comparison
evaluator = RegressionEvaluator(labelCol=label_column, metricName="rmse")

initial_model_test_rmse = evaluator.evaluate(initial_model.transform(test_df))
final_model_test_rmse = evaluator.evaluate(final_model.transform(test_df))

print(f"On the test data, the initial model achieved an RMSE of {initial_model_test_rmse} "
      f"and the final model achieved an RMSE of {final_model_test_rmse}.")

On the test data, the initial model achieved an RMSE of 0.6619734741026135 and the final model achieved an RMSE of 0.6166544409330698.


In [0]:
# página 22

# Conclusion

In this demo, we explored how to optimize machine learning models in Databricks using Optuna and HyperOpt with Spark ML. We demonstrated how these frameworks handle hyperparameter tuning at both the model training level and optimization level, leveraging distributed computing for scalability.

We compared multiple strategies for hyperparameter tuning:

* **Single-machine tuning using Optuna** for efficient local execution.
* **Distributed hyperparameter tuning using HyperOpt with Spark Trials** to scale across a cluster.
* **End-to-end SparkML tuning using `CrossValidator`** for native Spark-based optimization.

### Key Takeaways

* **Parallelization strategies** significantly impact model training efficiency and resource utilization.
* **Databricks provides multiple options** for hyperparameter tuning, allowing flexibility in balancing scalability vs. compute cost.
* **MLflow enables seamless experiment tracking**, making it easier to compare results across different tuning frameworks.

![image_1789438665052.png](./image_1789438665052.png "image_1789438665052.png")

In [0]:
# criado os experiments
#02d - Model Tuning with Hyperopt_spark
#02c - Model Tuning with spark cv
#02a - Model Tuning with Optuna_Spark